# AIAT 125 — Unit 1: AI Model Deployment Basics
## Guided Lab: Packaging, Loading, Validating & Benchmarking a scikit-learn Model

**Course**: AIAT 125 | **Institution**: Tuwaiq Academy, AI Diploma
**Unit**: 1 — Deployment Basics
**Total Points**: 100

---

### What You Will Do

This lab reinforces the exact skills from the six Unit 1 example notebooks. You take one trained scikit-learn model all the way through the early deployment lifecycle: **package → load & verify → serve → benchmark**.

| # | Task | Maps to example | Why it matters |
|---|------|-----------------|----------------|
| 1 | **Package a model + metadata** (`joblib`) | 02, 04 | The artifact is what production loads — it must carry its own metadata |
| 2 | **Load & verify the artifact** | 04, 05 | A deploy pipeline must reject corrupt/incomplete artifacts *before* serving |
| 3 | **In-process predict + validation gate** | 01, 05 | Validate inputs and prove accuracy clears a threshold before going live |
| 4 | **Latency benchmark (P50/P95/P99)** | 03 | An accurate model that is too slow still fails the user |

### How to Use This Notebook

- **Setup / Concept cells**: run them as-is — they install libraries and show the pattern you will reproduce.
- **Task cells**: look for `# TODO` comments and replace `None` / `pass` with your code.
- **Assertion cells**: run them after each task — they tell you immediately if something is wrong.
- **Final Gate**: the last cell prints PASS / FAIL for every task.

> **Tip**: If you get stuck, re-read the concept demo above the task — every task is a small variation of the demo just above it.

---


## Setup — Run This First

This installs the libraries (same stack as the example notebooks) and prepares a shared Iris train/test split that every task uses.


In [ ]:
# Run once; restart the kernel if packages were newly installed
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "joblib", "numpy"], check=False)

import joblib, json, time, os
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Use /tmp/ for all file I/O to avoid permission issues on any OS
SAVE_DIR = "/tmp/aiat125_unit1/"
os.makedirs(SAVE_DIR, exist_ok=True)

# Shared dataset — every task reuses this split (Iris: 4 features, 3 classes)
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)
FEATURE_NAMES = list(iris.feature_names)
CLASS_NAMES = [str(c) for c in iris.target_names]

print("Setup complete.")
print(f"Save directory : {SAVE_DIR}")
print(f"Train / test   : {len(X_train)} / {len(X_test)} samples")
print(f"Features       : {FEATURE_NAMES}")
print(f"Classes        : {CLASS_NAMES}")


---
## Concept 1 — Packaging a Model With Its Metadata

### Why bundle metadata with the model?

A bare `.joblib` file is just weights. Six months later nobody remembers which data it was trained on, who built it, or how accurate it was. The fix you saw in examples **02** and **04**: save the model **and** a metadata dict together in one bundle. Then the artifact is self-describing — a deploy pipeline can read the metadata without loading the model into a framework.

### The pattern (from examples 02 & 04)


In [ ]:
# ── CONCEPT DEMO — no changes needed ──────────────────────────────────────────
demo_model = RandomForestClassifier(n_estimators=10, random_state=0).fit(X_train, y_train)

demo_bundle = {
    "model": demo_model,
    "metadata": {
        "version": "demo-0.1",
        "developer": "AI_Team_Alpha",
        "accuracy": round(accuracy_score(y_test, demo_model.predict(X_test)), 4),
        "framework": f"scikit-learn",
        "feature_names": FEATURE_NAMES,
        "classes": CLASS_NAMES,
    },
}

demo_path = os.path.join(SAVE_DIR, "demo_bundle.joblib")
joblib.dump(demo_bundle, demo_path)

reloaded = joblib.load(demo_path)
print("Bundle saved ->", demo_path)
print("Top-level keys :", list(reloaded.keys()))
print("Metadata       :", reloaded["metadata"])


---
## Concept 2 — Loading and Verifying an Artifact Before You Serve It

### Why verify?

In examples **04** and **05** you saw that a deploy step never trusts a file blindly. Before serving, it checks the bundle actually contains a `model` and the metadata keys the service depends on. If anything is missing the pipeline must **fail loudly** (raise an error) rather than serve a half-broken model.

### The pattern (from examples 04 & 05)


In [ ]:
# ── CONCEPT DEMO — no changes needed ──────────────────────────────────────────
def verify_bundle(bundle):
    """Raise ValueError if the bundle is missing required structure."""
    for key in ["model", "metadata"]:
        if key not in bundle:
            raise ValueError(f"Bundle is missing top-level key: '{key}'")
    print("Bundle structure looks valid.")

verify_bundle(joblib.load(demo_path))   # passes

try:
    verify_bundle({"model": demo_model})   # no 'metadata' -> should fail
except ValueError as e:
    print("Caught expected error:", e)


---
## Concept 3 — Serving a Prediction In-Process (With Input Validation)

### Why validate the input?

Example **01** wrapped the model in an API and used a Pydantic schema to reject bad requests *before* they reach the model. Here we do the same idea in plain Python: a `predict` function that checks the input has exactly 4 numeric features, then returns the predicted class **and** a confidence score. Validating early gives clients a clear error instead of a confusing crash deep inside the model.

### The pattern (from example 01)


In [ ]:
# ── CONCEPT DEMO — no changes needed ──────────────────────────────────────────
def demo_predict(model, features):
    features = list(features)
    if len(features) != 4:
        raise ValueError(f"Expected 4 features, got {len(features)}")
    proba = model.predict_proba(np.array([features]))[0]
    idx = int(np.argmax(proba))
    return {"prediction": CLASS_NAMES[idx], "confidence": round(float(proba[idx]), 4)}

print(demo_predict(demo_model, [5.1, 3.5, 1.4, 0.2]))   # a setosa sample


---
## Concept 4 — Latency Benchmarking With Percentiles

### Why percentiles, not just the average?

Example **03** measured how long inference takes. Average latency hides outliers: if 95% of requests finish in 5 ms but 5% take 2 s, users feel an unreliable service. The industry standard:

- **P50** (median): the typical experience
- **P95**: the worst experience for 1 in 20 users — the usual SLA target
- **P99**: the worst experience for 1 in 100 users — early warning for tail latency

We use `np.percentile` (the same function as examples 03 and 05) so all notebooks agree on how percentiles are computed.

### The pattern (from example 03)


In [ ]:
# ── CONCEPT DEMO — no changes needed ──────────────────────────────────────────
def demo_latency(model, sample, iterations=50):
    latencies = []
    for _ in range(iterations):
        t0 = time.perf_counter()
        model.predict(sample)
        latencies.append((time.perf_counter() - t0) * 1000)   # ms
    return np.percentile(latencies, 50), np.percentile(latencies, 95)

one_sample = X_test[:1]
p50, p95 = demo_latency(demo_model, one_sample)
print(f"Demo latency  ->  P50={p50:.4f} ms | P95={p95:.4f} ms")


---
# Task 1 — Package a Trained Model With Metadata (25 points)

### Instructions

1. Train a `RandomForestClassifier(n_estimators=100, random_state=42)` on `X_train`, `y_train`. Call it **`task1_model`**.
2. Compute its accuracy on `X_test` / `y_test` with `accuracy_score`. Store it in **`task1_accuracy`**.
3. Build **`task1_metadata`** — a dict containing **exactly** these keys: `version`, `developer`, `accuracy`, `framework`, `feature_names`, `classes`. Set `accuracy` to `task1_accuracy`.
4. Save the bundle `{"model": task1_model, "metadata": task1_metadata}` to `TASK1_SAVE_PATH` with `joblib.dump`.

> This is exactly the packaging step from examples 02 and 04 — the bundle is the deployable artifact.


In [ ]:
# ── TASK 1 — SOLUTION ─────────────────────────────────────────────────────────
TASK1_SAVE_PATH = os.path.join(SAVE_DIR, "task1_model_bundle.joblib")

# 1a: train the model
task1_model = RandomForestClassifier(n_estimators=100, random_state=42)
task1_model.fit(X_train, y_train)

# 1b: test accuracy
task1_accuracy = round(accuracy_score(y_test, task1_model.predict(X_test)), 4)

# 1c: metadata with the six required keys
task1_metadata = {
    "version": "v1.0.0",
    "developer": "student",
    "accuracy": task1_accuracy,
    "framework": "scikit-learn",
    "feature_names": FEATURE_NAMES,
    "classes": CLASS_NAMES,
}

# 1d: save the bundle
joblib.dump({"model": task1_model, "metadata": task1_metadata}, TASK1_SAVE_PATH)

print(f"Task 1 complete. Test accuracy = {task1_accuracy}")
print(f"Bundle saved to: {TASK1_SAVE_PATH}")


In [ ]:
# ── ASSERTIONS: Task 1 ────────────────────────────────────────────────────────
assert isinstance(task1_model, RandomForestClassifier), "task1_model must be a RandomForestClassifier"
assert task1_model.n_estimators == 100, "use n_estimators=100"
assert task1_accuracy is not None and task1_accuracy > 0.8, f"test accuracy looks too low: {task1_accuracy}"

assert os.path.exists(TASK1_SAVE_PATH), f"bundle not found at {TASK1_SAVE_PATH}"
_bundle = joblib.load(TASK1_SAVE_PATH)
assert "model" in _bundle and "metadata" in _bundle, "bundle must have 'model' and 'metadata'"

for key in ["version", "developer", "accuracy", "framework", "feature_names", "classes"]:
    assert key in _bundle["metadata"], f"metadata is missing required key: '{key}'"
assert _bundle["metadata"]["accuracy"] == task1_accuracy, "metadata['accuracy'] must equal task1_accuracy"

print("Task 1 PASSED — model packaged with complete metadata.")


---
# Task 2 — Load and Verify the Artifact (25 points)

### Instructions

Implement `load_and_verify_artifact(path)` that a deploy pipeline would run:

1. Raise `FileNotFoundError` if `path` does not exist.
2. Load the bundle with `joblib.load`.
3. Raise `ValueError` if either `'model'` or `'metadata'` is missing.
4. Raise `ValueError` if `metadata` is missing any of `REQUIRED_METADATA_KEYS`.
5. Return the tuple `(model, metadata)`.

Then call it on `TASK1_SAVE_PATH` and store the results in `task2_model` and `task2_meta`.

> This is the gate from examples 04/05: never serve an artifact you have not verified.


In [ ]:
# ── TASK 2 — SOLUTION ─────────────────────────────────────────────────────────
REQUIRED_METADATA_KEYS = ["version", "developer", "accuracy", "framework", "feature_names", "classes"]

def load_and_verify_artifact(path):
    """Load a joblib bundle, verify its structure, return (model, metadata)."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"No artifact at {path}")

    bundle = joblib.load(path)

    for key in ["model", "metadata"]:
        if key not in bundle:
            raise ValueError(f"Bundle missing required key: '{key}'")

    for key in REQUIRED_METADATA_KEYS:
        if key not in bundle["metadata"]:
            raise ValueError(f"Metadata missing required key: '{key}'")

    return bundle["model"], bundle["metadata"]

task2_model, task2_meta = load_and_verify_artifact(TASK1_SAVE_PATH)
print("Loaded metadata:", task2_meta)


In [ ]:
# ── ASSERTIONS: Task 2 ────────────────────────────────────────────────────────
assert callable(load_and_verify_artifact), "load_and_verify_artifact must be a function"

# missing file -> FileNotFoundError
try:
    load_and_verify_artifact("/tmp/aiat125_unit1/does_not_exist.joblib")
    assert False, "should have raised FileNotFoundError"
except FileNotFoundError:
    pass

# bundle missing 'metadata' -> ValueError
_corrupt = os.path.join(SAVE_DIR, "corrupt_bundle.joblib")
joblib.dump({"model": task1_model}, _corrupt)
try:
    load_and_verify_artifact(_corrupt)
    assert False, "should have raised ValueError for missing 'metadata'"
except ValueError:
    pass

# metadata missing a required key -> ValueError
_incomplete = os.path.join(SAVE_DIR, "incomplete_bundle.joblib")
joblib.dump({"model": task1_model, "metadata": {"version": "x"}}, _incomplete)
try:
    load_and_verify_artifact(_incomplete)
    assert False, "should have raised ValueError for incomplete metadata"
except ValueError:
    pass

# successful load
assert isinstance(task2_model, RandomForestClassifier), "task2_model must be the loaded model"
assert isinstance(task2_meta, dict), "task2_meta must be a dict"
for key in REQUIRED_METADATA_KEYS:
    assert key in task2_meta, f"task2_meta missing key: '{key}'"

print("Task 2 PASSED — artifact loads and verification gate works.")


---
# Task 3 — In-Process Predict + Validation Gate (25 points)

### Instructions

**3a.** Implement `predict_one(model, features)`:
- Convert `features` to a list. If it does **not** have exactly 4 values, raise `ValueError`.
- If any value is not a number (`int`/`float`), raise `ValueError`.
- Otherwise return a dict `{"prediction": <class name>, "confidence": <float>}`, where the class name comes from `CLASS_NAMES` and confidence is the top `predict_proba` value (rounded to 4 dp).

**3b.** Run a **validation gate** (like example 05): predict on all of `X_test`, compute accuracy, store it in `val_accuracy`, and set `validation_passed = val_accuracy >= ACCURACY_THRESHOLD`.

Use `task2_model` (the artifact you just loaded and verified) for both parts.


In [ ]:
# ── TASK 3 — SOLUTION ─────────────────────────────────────────────────────────
ACCURACY_THRESHOLD = 0.85

def predict_one(model, features):
    """Validate the input, then return {'prediction', 'confidence'}."""
    features = list(features)
    if len(features) != 4:
        raise ValueError(f"Expected 4 features, got {len(features)}")
    if not all(isinstance(v, (int, float)) for v in features):
        raise ValueError("All features must be numeric (int or float)")

    proba = model.predict_proba(np.array([features]))[0]
    idx = int(np.argmax(proba))
    return {"prediction": CLASS_NAMES[idx], "confidence": round(float(proba[idx]), 4)}

# 3b: validation gate
val_accuracy = round(accuracy_score(y_test, task2_model.predict(X_test)), 4)
validation_passed = val_accuracy >= ACCURACY_THRESHOLD

print("Sample prediction:", predict_one(task2_model, [6.7, 3.0, 5.2, 2.3]))
print(f"Validation accuracy = {val_accuracy} | passed = {validation_passed}")


In [ ]:
# ── ASSERTIONS: Task 3 ────────────────────────────────────────────────────────
assert callable(predict_one), "predict_one must be a function"

# wrong number of features -> ValueError
try:
    predict_one(task2_model, [1.0, 2.0, 3.0])
    assert False, "should raise ValueError for != 4 features"
except ValueError:
    pass

# non-numeric feature -> ValueError
try:
    predict_one(task2_model, [5.1, "wide", 1.4, 0.2])
    assert False, "should raise ValueError for a non-numeric feature"
except ValueError:
    pass

# valid setosa-like sample
_out = predict_one(task2_model, [5.1, 3.5, 1.4, 0.2])
assert set(_out.keys()) == {"prediction", "confidence"}, "return dict must have exactly those two keys"
assert _out["prediction"] in CLASS_NAMES, "prediction must be a class name"
assert 0.0 <= _out["confidence"] <= 1.0, "confidence must be a probability"

# validation gate
assert val_accuracy is not None and val_accuracy >= ACCURACY_THRESHOLD, f"val_accuracy below threshold: {val_accuracy}"
assert validation_passed is True, "validation_passed must be True"

print("Task 3 PASSED — input validation and accuracy gate both work.")


---
# Task 4 — Latency Benchmark: P50 / P95 / P99 (25 points)

### Instructions

**4a.** Run exactly **100** single-sample predictions of `task2_model` on `X_test[:1]`, timing each with `time.perf_counter()`. Store the latencies (in **milliseconds**) in the list `latency_results`.

**4b.** Use `np.percentile` to compute `p50`, `p95`, `p99` from `latency_results` (the same function examples 03 and 05 use).

**4c.** The SLA is `p95 < 100` ms. A single sklearn prediction is microseconds, so this passes easily — but the assertion teaches you the gate.

> Optional: add a short 5-iteration warm-up loop (untimed) before the timed loop to remove cold-start noise.


In [ ]:
# ── TASK 4 — SOLUTION ─────────────────────────────────────────────────────────
NUM_ITERATIONS = 100
one_sample = X_test[:1]

# warm-up (untimed)
for _ in range(5):
    task2_model.predict(one_sample)

# timed loop
latency_results = []
for _ in range(NUM_ITERATIONS):
    t0 = time.perf_counter()
    task2_model.predict(one_sample)
    latency_results.append((time.perf_counter() - t0) * 1000)

p50 = np.percentile(latency_results, 50)
p95 = np.percentile(latency_results, 95)
p99 = np.percentile(latency_results, 99)

print(f"Latency over {NUM_ITERATIONS} predictions:")
print(f"  P50 = {p50:.4f} ms | P95 = {p95:.4f} ms | P99 = {p99:.4f} ms")


In [ ]:
# ── ASSERTIONS: Task 4 ────────────────────────────────────────────────────────
assert isinstance(latency_results, list), "latency_results must be a list"
assert len(latency_results) == NUM_ITERATIONS, f"need {NUM_ITERATIONS} timings, got {len(latency_results)}"
assert all(isinstance(x, float) and x > 0 for x in latency_results), "all latencies must be positive floats (ms)"

assert p50 is not None and p95 is not None and p99 is not None, "compute p50, p95, p99"
assert p50 <= p95 <= p99, f"percentile ordering must hold: {p50:.4f} <= {p95:.4f} <= {p99:.4f}"
assert p95 < 100, f"p95 latency {p95:.4f} ms exceeds the 100 ms SLA — add a warm-up loop"

print("Task 4 PASSED — latency benchmark complete, P95 < 100 ms.")


---
# Final Deployment Gate — Summary Report

Run this cell last. It checks all four tasks and prints a PASS / FAIL report. All four must PASS before you submit.


In [ ]:
# ── DEPLOYMENT GATE — Final Summary ──────────────────────────────────────────
results = {}

try:
    assert isinstance(task1_model, RandomForestClassifier) and task1_accuracy > 0.8
    _b = joblib.load(TASK1_SAVE_PATH)
    assert "model" in _b and "metadata" in _b
    for k in ["version", "developer", "accuracy", "framework", "feature_names", "classes"]:
        assert k in _b["metadata"]
    results["Task 1 - Package model + metadata (25 pts)"] = "PASS"
except Exception as e:
    results["Task 1 - Package model + metadata (25 pts)"] = f"FAIL - {e}"

try:
    assert callable(load_and_verify_artifact)
    assert isinstance(task2_model, RandomForestClassifier) and isinstance(task2_meta, dict)
    results["Task 2 - Load & verify artifact (25 pts)"] = "PASS"
except Exception as e:
    results["Task 2 - Load & verify artifact (25 pts)"] = f"FAIL - {e}"

try:
    assert callable(predict_one)
    assert validation_passed is True and val_accuracy >= ACCURACY_THRESHOLD
    results["Task 3 - Predict + validation gate (25 pts)"] = "PASS"
except Exception as e:
    results["Task 3 - Predict + validation gate (25 pts)"] = f"FAIL - {e}"

try:
    assert len(latency_results) == 100
    assert p50 <= p95 <= p99 and p95 < 100
    results["Task 4 - Latency benchmark P50/P95/P99 (25 pts)"] = "PASS"
except Exception as e:
    results["Task 4 - Latency benchmark P50/P95/P99 (25 pts)"] = f"FAIL - {e}"

print("=" * 60)
print("  AIAT 125 - Unit 1 Guided Lab: DEPLOYMENT GATE REPORT")
print("=" * 60)
passed = 0
for task, status in results.items():
    icon = "PASS" if status == "PASS" else "FAIL"
    print(f"  [{icon}] {task}: {status}")
    if status == "PASS":
        passed += 1
print("=" * 60)
print(f"  Score: {passed * 25} / 100  ({passed}/4 tasks passed)")
print("=" * 60)
print("  ALL GATES PASSED - cleared for deployment." if passed == 4
      else "  GATES NOT FULLY PASSED - fix the failing tasks and re-run.")


---
## Closing Takeaway

Answer in your own words (no code needed):

1. **Packaging**: What other fields would you add to the metadata to make a rollback decision easier in a team setting?
2. **Verification**: Why should the deploy pipeline *raise an error* on a bad artifact instead of just printing a warning and continuing?
3. **Validation gate**: Your validation accuracy is high but real users complain. What does that suggest about your test set?
4. **Latency**: If P99 is 800 ms but P95 is 40 ms, what does that gap tell you about your system?
5. **Lifecycle**: Put these four tasks in the order a model travels through deployment, and name what could go wrong at each step.

---
**Next**: continue with the Unit 2 notebooks (serving frameworks, versioning, batch vs real-time).

**References:**
- [scikit-learn — Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- [joblib documentation](https://joblib.readthedocs.io/)
- [Google SRE Book — Monitoring & percentiles](https://sre.google/sre-book/monitoring-distributed-systems/)
